# Library Generator

This notebook generates a library of curcumin analogs by substituting R-groups on a core scaffold.
It uses a defined pool of radicals and combinatorial generation, filters results using Lipinski's Rule of 5 (Ro5), and saves valid molecules to a SMILES file.

In [1]:
from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, AllChem, Lipinski, Descriptors
from IPython.display import SVG, display
from pathlib import Path
from loguru import logger
import numpy as np
from tqdm.auto import tqdm
from datetime import datetime
from itertools import product
from pprint import pprint
import io
import time
import sys 
import os

sys.path.append("..")
from utils import draw

## Configuration

Define the pool of available radicals (R-groups) and generate all possible combinations.
We use 4 substitution sites, creating a combinatorial space.

In [2]:
radicals_pool = {
    "CH3": "C",
    "CH3OH": "CO",
    "OMe": "OC",
    "OH": "O",
    "H": "[H]",
    "NH2": "N",
    "NO2": "N(=O)=O",
    "NCH3": "NC",
    "SO3H": "S(=O)(=O)O",
    "SO2Me": "S(=O)(=O)C",
    "F": "F",
    "Cl": "Cl"
}

all_combinations = list(product(radicals_pool.keys(), repeat=4))
print(f"Rads count: {len(radicals_pool)}")
print(f"Combs count: {len(all_combinations):,}")

## Load Scaffold

Load the core scaffold with R-group placeholders prepared in the previous notebook.

In [3]:
mol_folder = Path("..") / "data" 

with open(mol_folder / "raw" / "curcumin_scaffold.smi", "r") as f:
    smiles = f.read()

rad_mol = Chem.MolFromSmiles(smiles)
draw(rad_mol)

## Filtering and Generation Functions

- `check_lipinski`: Validates if a molecule follows Lipinski's Rule of 5 (max 1 violation).
- `build_analog`: Replaces R-groups on the scaffold using reaction SMARTS and validates the result.

In [4]:
def check_lipinski(mol):
    if mol is None:
        return False
        
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)

    violations = sum([
        mw > 500, 
        logp > 5, 
        hbd > 5, 
        hba > 10
    ])
    
    return violations <= 1

def build_analog(base_mol, comb, radicals_pool):
    work_mol = Chem.Mol(base_mol) 
    
    for i, group_key in enumerate(comb):
        group_smiles = radicals_pool.get(group_key)
        
        if not group_smiles:
            logger.error(f"Group {group_key} not found.")
            continue
        
        rxn_smarts = f'[*:1]-[#{100+i}]>>[*:1]-{group_smiles}'
        rxn = AllChem.ReactionFromSmarts(rxn_smarts)
        
        products = rxn.RunReactants((work_mol,))
        
        if products and products[0]:
            work_mol = products[0][0]
        else:
            logger.warning(f"Reaction failed for R-group {group_key}.")
            return None, None 

    try:
        Chem.SanitizeMol(work_mol)
        
        if not check_lipinski(work_mol):
            return None, None

        smiles_string = Chem.MolToSmiles(work_mol)
        return smiles_string, comb
        
    except Exception as e:
        logger.error(f"Sanitization or Ro5 error: {e}")
        return None, None

## Execution

Iterate through all combinations, generate molecules, apply filters, and save the results to `ligands_list.smi`.
Uses a buffer to improve file I/O performance.

In [5]:
file_path = mol_folder / "interim" / "ligands_list.smi"
BATCH_SIZE = 10000

start_time = time.time()
counter_total = 0
counter_written = 0
buffer = io.StringIO()

if input("Are you sure? [y/n] ").lower() == "y":
    logger.info(f"Starting generation. Batch size: {BATCH_SIZE}")

    try:
        with open(file_path, 'w') as outfile:
            for comb in tqdm(all_combinations):
                smiles, comb = build_analog(rad_mol, comb, radicals_pool)
                counter_total += 1
                
                if smiles:
                    buffer.write(f"""{"_".join(comb)} {smiles}\n""")
                    counter_written += 1
                    
                    if counter_written % BATCH_SIZE == 0:
                        outfile.write(buffer.getvalue())
                        buffer.seek(0)
                        buffer.truncate(0)

            outfile.write(buffer.getvalue())

    except Exception as e:
        logger.critical(f"Critical error in main loop: {e}")

    finally:
        end_time = time.time()
        logger.info("--- FINISHED ---")
        logger.info(f"Total time: {end_time - start_time:.2f} seconds.")
        logger.info(f"Total combinations processed: {counter_total}")
        logger.info(f"Molecules passing Ro5 and saved: {counter_written}")
        if counter_total > 0:
            logger.info(f"Ro5 Efficiency: {(counter_written / counter_total) * 100:.2f}%")
else:
    print("Aborted by user")